<div dir="rtl" style="text-align:right">
<h1>ساختِ صدای فارسی با هوش مصنوعی، در گوگل کولب</h1>
<p><b>کانال میکرولرن</b></p>
<p>این نوت‌بوک <b>متن فارسی را به صدا تبدیل می‌کند</b>، با مدل Chatterbox و نسخهٔ فارسیِ آن (<code dir="ltr">Thomcles/Chatterbox-TTS-Persian-Farsi</code>)،</p>
<p>با صدای پیش‌فرض یا با یک نمونهٔ کوتاه از صدای خودتان.</p>
<p><b>قبل از شروع:</b> از منوی بالا <code dir="ltr">Runtime → Change runtime type</code> را باز کنید و <code dir="ltr">T4 GPU</code> را انتخاب کنید.</p>
<p>بعد سلول‌ها را <b>به ترتیب</b>، یکی‌یکی اجرا کنید (دکمهٔ ▶ کنار هر سلول یا <code dir="ltr">Shift+Enter</code>).</p>
<blockquote>مجوزِ مدلِ فارسی CC BY-NC 4.0 است: فقط برای استفادهٔ شخصی و غیرتجاری.</blockquote>
<blockquote>فقط صدای خودتان یا صدای کسی را که اجازه داده بسازید.</blockquote>
</div>

<div dir="rtl" style="text-align:right">
<h2>۱. بررسیِ کارت گرافیک</h2>
</div>

In [ ]:
# این سلول نشان می‌دهد کولب به ما کارت گرافیک داده یا نه.
# اگر خطا داد یعنی GPU روشن نیست: Runtime → Change runtime type → T4 GPU
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print("GPU آماده است:", torch.cuda.is_available())
import numpy, transformers
print("torch", torch.__version__, "| numpy", numpy.__version__, "| transformers", transformers.__version__)

<div dir="rtl" style="text-align:right">
<h2>۲. نصبِ Chatterbox</h2>
<p>حدود یکی دو دقیقه طول می‌کشد. پیام‌های قرمزِ pip دربارهٔ نسخه‌ها معمولاً مشکلی نیستند.</p>
</div>

In [ ]:
# خودِ Chatterbox را بدون وابستگی‌هایش نصب می‌کنیم تا PyTorchِ آمادهٔ کولب عوض نشود
# (نسخهٔ ثابت از گیت‌هاب، همان نسخه‌ای که ما تست کردیم)
!pip install -q --no-deps "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@ab55923c27757c90cfcc407baa57dc25a7d25217"
# وابستگی‌هایی که کولب از قبل ندارد
!pip install -q s3tokenizer conformer==0.3.2 diffusers==0.29.0 omegaconf pyloudnorm spacy-pkuseg pykakasi==2.3.0 \
    "resemble-perth @ git+https://github.com/resemble-ai/Perth.git@master"
print("نصب تمام شد")

<div dir="rtl" style="text-align:right">
<h2>۳. ورود به Hugging Face (از بخشِ Secrets)</h2>
<p>مدلِ فارسی روی Hugging Face است و قبل از دانلود باید شرایطش را بپذیرید:</p>
<ol>
<li>در huggingface.co حساب بسازید و وارد شوید.</li>
<li>صفحهٔ مدل <code dir="ltr">Thomcles/Chatterbox-TTS-Persian-Farsi</code> را باز کنید و دسترسی را تأیید کنید.</li>
<li>در تنظیمات حساب، بخشِ Access Tokens، یک توکن از نوع <b>Read</b> بسازید.</li>
<li>در کولب، روی آیکونِ کلید 🔑 (Secrets) در نوار سمت چپ بزنید، یک Secret به اسمِ <code dir="ltr">HF_TOKEN</code> بسازید، توکن را در Value بگذارید و دسترسیِ نوت‌بوک را روشن کنید.</li>
</ol>
<p>توکن را <b>هیچ‌وقت</b> مستقیم داخلِ کد ننویسید.</p>
</div>

In [ ]:
# توکن را از Secretsِ کولب می‌خوانیم؛ خودِ توکن هیچ‌جا چاپ نمی‌شود
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("توکن خوانده شد")

<div dir="rtl" style="text-align:right">
<h2>۴. دانلودِ مدلِ فارسی</h2>
</div>

In [ ]:
# فایلِ مدلِ فارسی (حدود ۲ گیگابایت) را از Hugging Face می‌گیریم
from huggingface_hub import hf_hub_download
T3_FA = hf_hub_download("Thomcles/Chatterbox-TTS-Persian-Farsi", "t3_fa.safetensors")
print("دانلود شد:", T3_FA)

<div dir="rtl" style="text-align:right">
<h2>۵. بارگذاریِ مدل و ساختِ اولین جملهٔ فارسی</h2>
</div>

In [ ]:
# مدلِ چندزبانهٔ Chatterbox را بار می‌کنیم و بخشِ T3ِ آن را با نسخهٔ فارسی عوض می‌کنیم
import re, random
import numpy as np
import torch
from IPython.display import Audio, display
from safetensors.torch import load_file
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
model.t3.load_state_dict(load_file(T3_FA, device="cuda"))
model.t3.to("cuda").eval()
DEFAULT_VOICE = model.conds  # صدای پیش‌فرضِ مدل را نگه می‌داریم


def split_text(text, limit=200):
    # هر بار ساخت حداکثر حدود ۴۰ ثانیه صدا می‌دهد، پس متنِ بلند را سرِ جمله‌ها تکه می‌کنیم
    parts = [p.strip() for p in re.split(r"(?<=[.؟?!؛])\s+|\n+", text) if p.strip()]
    pieces = []
    for p in parts:  # جملهٔ خیلی بلند را سرِ فاصلهٔ کلمه‌ها می‌بُریم
        while len(p) > limit:
            cut = p.rfind(" ", 0, limit)
            cut = cut if cut > 0 else limit
            pieces.append(p[:cut].strip())
            p = p[cut:].strip()
        if p:
            pieces.append(p)
    chunks, cur = [], ""
    for p in pieces:  # جمله‌های کوتاه را کنارِ هم می‌گذاریم تا تکه‌ها زیادی کوچک نباشند
        if cur and len(cur) + 1 + len(p) > limit:
            chunks.append(cur)
            cur = p
        else:
            cur = f"{cur} {p}".strip()
    if cur:
        chunks.append(cur)
    return chunks


def say(text, voice=None, exaggeration=0.6, cfg_weight=0.5, seed=42):
    # متن را به صدا تبدیل می‌کند؛ voice = مسیرِ یک نمونهٔ صدا، یا None برای صدای پیش‌فرض
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if voice:
        model.prepare_conditionals(voice, exaggeration=exaggeration)
    else:
        model.conds = DEFAULT_VOICE
    gap = np.zeros(int(0.15 * model.sr), dtype=np.float32)  # مکثِ کوتاه بینِ تکه‌ها
    pieces = []
    for chunk in split_text(text):
        wav = model.generate(chunk, language_id=None, exaggeration=exaggeration,
                             cfg_weight=cfg_weight, temperature=0.7, top_p=0.5)
        pieces += [wav.squeeze().cpu().numpy().astype(np.float32), gap]
    return np.concatenate(pieces[:-1])


print("مدل آماده است")

In [ ]:
# اولین جمله با صدای پیش‌فرضِ مدل
text = "سلام! این صدا رو هوش مصنوعی ساخته، از روی متنی که خودم نوشتم."
wav = say(text)
display(Audio(wav, rate=model.sr))

<div dir="rtl" style="text-align:right">
<h2>۶. ساختن با صدای خودتان (فقط از یک نمونهٔ کوتاه)</h2>
<p>یک فایلِ صوتیِ <b>۱۰ تا ۱۵ ثانیه</b> از صدای خودتان آپلود کنید: یک نفر، بدون موسیقی، در اتاقِ ساکت.</p>
<p>لحنِ این نمونه روی لحنِ خروجی اثر می‌گذارد: اگر آرام و دوستانه حرف بزنید، خروجی هم همان‌طور می‌شود.</p>
</div>

In [ ]:
# پنجرهٔ انتخابِ فایل باز می‌شود؛ نمونهٔ صدایتان را انتخاب کنید (wav یا mp3)
from google.colab import files
uploaded = files.upload()
MY_VOICE = list(uploaded)[0]
print("نمونهٔ صدا:", MY_VOICE)
display(Audio(MY_VOICE))

In [ ]:
# همان جمله، این بار با صدای شما
wav = say(text, voice=MY_VOICE)
display(Audio(wav, rate=model.sr))

<div dir="rtl" style="text-align:right">
<h2>۷. متنِ بلند</h2>
<p>متن را هر قدر خواستید بلند بنویسید؛ خودش سرِ جمله‌ها تکه می‌شود و تکه‌ها پشتِ هم چسبانده می‌شوند.</p>
<ul>
<li>علامت‌گذاری مهم است: «.» آخرِ جمله لحن را پایین می‌آورد و «؟» بالا می‌برد.</li>
<li><code dir="ltr">exaggeration</code> بیشتر (مثلاً 0.8) صدا را پرانرژی‌تر می‌کند، کمتر (مثلاً 0.4) آرام‌تر.</li>
</ul>
</div>

In [ ]:
long_text = """
امروز می‌خوام یه چیز جالب نشونتون بدم. چند وقت پیش داشتم فکر می‌کردم که چقدر خوب می‌شد اگه می‌تونستم متن‌هام رو بدون ضبط کردن، با صدای خودم بشنوم.
اولش فکر می‌کردم این کار فقط از شرکت‌های بزرگ برمیاد. ولی الان با یه مدل رایگان و یه کارت گرافیک، خیلی راحت‌تر از اون چیزیه که فکرش رو می‌کنید.
شما چی فکر می‌کنید؟ به نظرتون صدا طبیعی شده؟
"""
wav = say(long_text, voice=MY_VOICE, exaggeration=0.6)
import soundfile as sf
sf.write("long.wav", wav, model.sr)
print(f"طول صدا: {len(wav) / model.sr:.1f} ثانیه")
display(Audio(wav, rate=model.sr))

In [ ]:
# فایل را روی کامپیوترتان دانلود کنید
files.download("long.wav")